# CalledIt — Prompt Evaluation Explorer

Interactive dashboard for analyzing how prompt changes affect the agent chain across predictions at varying levels of fuzziness.

**Key questions this notebook answers:**
1. How does any change to a prompt (or collection of prompts) affect success across all prediction types?
2. For fuzzy predictions, how effective is the HITL clarification loop at producing accurate, verifiable predictions?
3. Which agent is the bottleneck? Where should we focus prompt tuning effort?
4. What alternative decisions could we have made, and what does the data suggest?

In [ ]:
import json, glob, os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from IPython.display import display, HTML

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

# Load all data
with open('score_history.json') as f:
    history = json.load(f)

reports = {}
for path in sorted(glob.glob('reports/eval-*.json')):
    with open(path) as f:
        reports[os.path.basename(path)] = json.load(f)

print(f'Loaded {len(history["evaluations"])} eval runs, {len(reports)} detailed reports')

## 1. Score Trends Over Time
Each point represents an eval run. Annotations show which prompt versions were active.

In [ ]:
evals = history['evaluations']
df_trends = pd.DataFrame([
    {
        'run': i+1,
        'timestamp': e['timestamp'][:16],
        'pass_rate': e['overall_pass_rate'] * 100,
        'prompts': ', '.join(f"{k}:{v}" for k,v in e.get('prompt_version_manifest',{}).items()),
        **{f'cat_{k}': v*100 for k,v in e.get('per_category_accuracy',{}).items()}
    }
    for i, e in enumerate(evals)
])

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df_trends['run'], df_trends['pass_rate'], 'o-', linewidth=2, markersize=10, label='Overall Pass Rate', color='#2d6a4f')
for col in [c for c in df_trends.columns if c.startswith('cat_')]:
    ax.plot(df_trends['run'], df_trends[col], 's--', markersize=6, label=col.replace('cat_',''), alpha=0.7)
ax.set_xlabel('Eval Run')
ax.set_ylabel('Score (%)')
ax.set_title('Eval Score Trends — How Prompt Changes Affect Quality')
ax.legend()
ax.set_ylim(0, 105)
ax.axhline(y=80, color='orange', linestyle='--', alpha=0.3, label='80% target')
plt.tight_layout()
plt.show()

## 2. Latest Run — Test Case Heatmap
Rows = test cases, Columns = evaluators. Red cells = problems to investigate.

In [ ]:
latest_name = sorted(reports.keys())[-1]
latest = reports[latest_name]
print(f'Analyzing: {latest_name}')
print(f'Pass rate: {latest["overall_pass_rate"]*100:.0f}% ({latest["passed"]}/{latest["total_tests"]})')

# Build heatmap dataframe
all_evals = set()
for tc in latest['per_test_case_scores']:
    all_evals.update(tc.get('evaluator_scores', {}).keys())
evaluators = sorted(all_evals)

rows = []
for tc in latest['per_test_case_scores']:
    row = {'test_case': f"{tc['test_case_id']} ({tc['layer']})"}
    for ev in evaluators:
        row[ev] = tc.get('evaluator_scores', {}).get(ev, {}).get('score', float('nan'))
    rows.append(row)

df_heat = pd.DataFrame(rows).set_index('test_case')
fig, ax = plt.subplots(figsize=(max(14, len(evaluators)*1.5), max(8, len(rows)*0.45)))
sns.heatmap(df_heat, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1, linewidths=0.5, ax=ax)
ax.set_title(f'Score Heatmap — {latest_name}')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. HITL Clarification Effectiveness
For fuzzy predictions: How well did the ReviewAgent's questions + user clarifications improve the prediction?

In [ ]:
fuzzy = [tc for tc in latest['per_test_case_scores'] if tc['layer'] == 'fuzzy']
if fuzzy:
    df_fuzzy = pd.DataFrame([
        {
            'test_case': tc['test_case_id'],
            'Convergence': tc['evaluator_scores'].get('Convergence', {}).get('score', 0),
            'Clarification Quality': tc['evaluator_scores'].get('R1_ClarificationQuality', {}).get('score', 0),
            'Post-Clarification Category': tc['evaluator_scores'].get('R2_CategoryMatch', {}).get('score', 0),
        }
        for tc in fuzzy
    ]).set_index('test_case')
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('HITL Clarification Effectiveness', fontsize=13, fontweight='bold')
    
    for i, col in enumerate(df_fuzzy.columns):
        colors = ['#2d6a4f' if v >= 0.5 else '#d62828' for v in df_fuzzy[col]]
        df_fuzzy[col].plot(kind='bar', ax=axes[i], color=colors)
        axes[i].set_title(col)
        axes[i].set_ylim(0, 1.1)
        axes[i].axhline(y=0.5, color='orange', linestyle='--', alpha=0.5)
        axes[i].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    display(HTML('<h4>Key Insight</h4>'))
    avg_conv = df_fuzzy['Convergence'].mean()
    avg_cq = df_fuzzy['Clarification Quality'].mean()
    avg_r2 = df_fuzzy['Post-Clarification Category'].mean()
    print(f'Avg Convergence: {avg_conv:.2f} — {"Good" if avg_conv > 0.7 else "Needs improvement" if avg_conv > 0.5 else "Poor"}')
    print(f'Avg Clarification Quality: {avg_cq:.2f} — {"Good" if avg_cq > 0.7 else "Keyword matching too brittle" if avg_cq > 0.3 else "ReviewAgent questions miss expected topics"}')
    print(f'Avg Post-Clarification Category Match: {avg_r2:.2f} — {"Excellent" if avg_r2 > 0.9 else "Good" if avg_r2 > 0.7 else "Category not converging after clarification"}')

## 4. Run Comparison — What Changed?
Select two runs to compare side-by-side.

In [ ]:
if len(evals) >= 2:
    prev = evals[-2]
    curr = evals[-1]
    
    print(f'Comparing: {prev["timestamp"][:16]} → {curr["timestamp"][:16]}')
    print(f'Pass rate: {prev["overall_pass_rate"]*100:.0f}% → {curr["overall_pass_rate"]*100:.0f}%')
    print()
    
    # Category deltas
    all_cats = set(list(prev.get('per_category_accuracy',{}).keys()) + list(curr.get('per_category_accuracy',{}).keys()))
    for cat in sorted(all_cats):
        p = prev.get('per_category_accuracy',{}).get(cat, 0) * 100
        c = curr.get('per_category_accuracy',{}).get(cat, 0) * 100
        delta = c - p
        icon = '↑' if delta > 0 else '↓' if delta < 0 else '='
        print(f'  {cat}: {p:.0f}% → {c:.0f}% {icon} ({delta:+.0f}%)')
    
    # Prompt version changes
    pm_prev = prev.get('prompt_version_manifest', {})
    pm_curr = curr.get('prompt_version_manifest', {})
    changes = {k: (pm_prev.get(k,'?'), pm_curr.get(k,'?')) for k in set(list(pm_prev)+list(pm_curr)) if pm_prev.get(k) != pm_curr.get(k)}
    if changes:
        print(f'\nPrompt changes: {json.dumps(changes, indent=2)}')
    else:
        print('\nNo prompt version changes between runs.')
else:
    print('Need at least 2 eval runs for comparison.')

## 5. Decision Analysis — What If?
Explore how different thresholds or category boundaries would change results.

In [ ]:
# What if we changed the pass threshold?
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
pass_rates_by_threshold = []

for thresh in thresholds:
    passed = 0
    for tc in latest['per_test_case_scores']:
        scores = tc.get('evaluator_scores', {})
        all_pass = all(s.get('score', 0) >= thresh for s in scores.values() if isinstance(s, dict) and 'score' in s)
        if all_pass:
            passed += 1
    pass_rates_by_threshold.append(passed / len(latest['per_test_case_scores']) * 100)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, pass_rates_by_threshold, 'o-', linewidth=2, markersize=8, color='#2d6a4f')
ax.set_xlabel('Pass Threshold (all evaluator scores must exceed this)')
ax.set_ylabel('Pass Rate (%)')
ax.set_title('Sensitivity Analysis — How Pass Threshold Affects Results')
ax.axvline(x=0.5, color='orange', linestyle='--', alpha=0.5, label='Current threshold (0.5)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nThis shows how sensitive our pass rate is to the threshold choice.')
print('A steep drop means many test cases are near the boundary — fragile.')
print('A gradual slope means robust performance across thresholds.')